In [22]:
import os
import re
import csv
import pandas as pd

In [23]:
# Rutas a las carpetas
carpeta_pls = '../datos/Cochrane/train/pls'
carpeta_non_pls = '../datos/Cochrane/train/non_pls'

In [24]:
# Leer archivos en las carpetas
archivos_pls = [f for f in os.listdir(carpeta_pls) if os.path.isfile(os.path.join(carpeta_pls, f))] if os.path.exists(carpeta_pls) else []
archivos_non_pls = [f for f in os.listdir(carpeta_non_pls) if os.path.isfile(os.path.join(carpeta_non_pls, f))] if os.path.exists(carpeta_non_pls) else []

In [25]:
# Limpiar archivos: eliminar los que tienen accumulated_section o _section
archivos_limpios_pls = [f for f in archivos_pls if "accumulated_section" not in f and "_section" not in f]
archivos_limpios_non_pls = [f for f in archivos_non_pls if "accumulated_section" not in f and "_section" not in f]

In [26]:
# Función para extraer el código CDXXXXXX
def extraer_codigo(nombre_archivo):
    match = re.search(r'CD\d{6}', nombre_archivo)
    return match.group(0) if match else None
# Mapear archivos a sus códigos
dic_pls = {extraer_codigo(f): f for f in archivos_limpios_pls if extraer_codigo(f)}
dic_non_pls = {extraer_codigo(f): f for f in archivos_limpios_non_pls if extraer_codigo(f)}
# Encontrar códigos comunes
codigos_comunes = set(dic_pls.keys()) & set(dic_non_pls.keys())
# Guardar datos en lista para luego convertir en DataFrame
filas = []

In [27]:
for codigo in sorted(codigos_comunes):
    path_non_pls = os.path.join(carpeta_non_pls, dic_non_pls[codigo])
    path_pls = os.path.join(carpeta_pls, dic_pls[codigo])

    try:
        with open(path_non_pls, 'r', encoding='utf-8') as f1, open(path_pls, 'r', encoding='utf-8') as f2:
            texto_non_pls = f1.read().strip()
            texto_pls = f2.read().strip()
            filas.append({
                'codigo_comun': codigo,
                'non_pls': texto_non_pls,
                'pls': texto_pls
            })
    except Exception as e:
        print(f"Error procesando {codigo}: {e}")

In [28]:
# Crear DataFrame con pandas
df = pd.DataFrame(filas, columns=['codigo_comun', 'non_pls', 'pls'])

# Guardar como Excel
ruta_excel = '../datos/Cochrane/train/train.xlsx'
df.to_excel(ruta_excel, index=False)
print(f"Excel generado en: {ruta_excel}")

Excel generado en: ../datos/Cochrane/train/train.xlsx
